# Part 2 · Notebook 08 — Performance and risk metrics

**Sessions:** S8 (Performance & risk metrics) · Clinic W2 · [Lesson plan](../../docs/lessons/PART_02_QUANT_TOOLKIT.md)

**You will:**
1. Compute Sharpe, Sortino, drawdown, VaR and CVaR yourself.
2. Build a tear-sheet table for 10 tickers.
3. Compare VaR methods and see the Sharpe-annualization pitfall.

How these notebooks work: the loading and plotting code is written for you. Cells marked **✍️ Your turn** need 1–5 lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p2lib.py is in notebooks/part02/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p2lib as p

p.use_course_style()
pd.set_option("display.float_format", "{:,.4f}".format)
prices = p.load_prices()          # dates × 10 tickers (course data via P2_DATA, else synthetic)
rets = p.log_returns(prices)      # daily log returns
prices.tail(3)

In [ ]:
simple = p.simple_returns(prices)          # metrics use SIMPLE returns (they add across assets)
r = simple["SPY"]
RF = 0.02                                    # ✏️ risk-free rate (annual)

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
# ✍️ annualized Sharpe: mean(excess) / std(excess) × sqrt(252), with excess = r − RF/252
sharpe = ...
sharpe = p.check("Sharpe", sharpe, p.sharpe(r, RF))

In [ ]:
equity = (1 + r).cumprod()
# ✍️ maximum drawdown: min over time of equity / running maximum − 1
mdd = ...
mdd = p.check("maximum drawdown", mdd, p.max_drawdown(equity))

In [ ]:
# ✍️ historical 99% VaR (the loss exceeded on 1% of days, as a positive number) and CVaR (average loss beyond it)
var99 = ...
cvar99 = ...
var99, cvar99 = p.check("VaR and CVaR (99%)", (var99, cvar99), p.hist_var_cvar(r, 0.99))
print(f"99% one-day VaR {var99:.2%}, CVaR {cvar99:.2%}")

## Tear-sheet table for all tickers

In [ ]:
table = pd.DataFrame({t: p.metrics(simple[t], RF, benchmark=simple["SPY"]) for t in simple}).T
table.style.format("{:.2f}").format("{:.1%}", subset=["CAGR", "Volatility", "Max drawdown", "VaR 99%", "CVaR 99%", "Alpha (ann.)"])

In [ ]:
TICKER = "AAPL"     # ✏️ change me
eq = (1 + simple[TICKER]).cumprod()
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
eq.plot(ax=axes[0], title=f"{TICKER}: growth of $1")
((eq / eq.cummax() - 1) * 100).plot(ax=axes[1], title="Drawdown (%)", color=p.PALETTE[1])
for ax in axes: ax.set_xlabel("")
plt.tight_layout(); plt.show()

## VaR methods: historical vs normal vs Cornish–Fisher

In [ ]:
from scipy import stats
mu, sd, s, k = r.mean(), r.std(), stats.skew(r), stats.kurtosis(r)
z = stats.norm.ppf(0.01)
zcf = z + (z**2 - 1) * s / 6 + (z**3 - 3 * z) * k / 24 - (2 * z**3 - 5 * z) * s**2 / 36
pd.Series({"historical": var99, "normal": -(mu + z * sd), "Cornish–Fisher": -(mu + zcf * sd)},
          name="99% one-day VaR").map("{:.2%}".format)

## The annualization pitfall: smoothed returns inflate Sharpe

In [ ]:
smoothed = 0.5 * r + 0.5 * r.shift(1)            # e.g. stale prices of an illiquid asset
for name, x in {"SPY": r, "smoothed SPY": smoothed.dropna()}.items():
    monthly = (1 + x).resample("ME").prod() - 1
    print(f"{name:<14} Sharpe from daily {p.sharpe(x, RF):.2f}  |  from monthly {p.sharpe(monthly, RF, periods=12):.2f}"
          f"  |  daily autocorrelation {x.autocorr():.2f}")

## Questions
1. Which ticker has the best Sharpe but the worst drawdown? What does that say about using one metric?
2. When does the normal VaR underestimate risk the most?
3. Why does smoothing raise the daily-based Sharpe but not the monthly one?

**Graded (Part 2, B):** turn these formulas into a tested `metrics` module (see the lesson plan, Section 7).